In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from create_stream import CreateStream

In [ ]:
q = b = 89
l = t = 15
m = 2**t
n = m 

# Del 1

In [ ]:
#Opgave1
#a)
def multiply_shift(x, l):
    a = 0xbe457450e667e943 & 0x1FFFFFFFFFFFFFFF
    return ((a * x)&0xFFFFFFFFFFFFFFFF) >> (64 - l)

#b)
def multiply_mod_prime(x, q, l):
    p = 2**q - 1
    a = 0x579a66a9030f158bb0fe1e9e & p
    b = 0xe4f64a81b0b65822808f0cae & p

    def f(x):
        y = (x & p) + (x >> q)
        if (y>=p):
            y-=p
        return y

    y = f(x * a) + b
    if y >= p:
        y =- p 
    return y & ((2**l) - 1)

#c)
def test(q, l):
    A = 0
    B = 0
    stream = CreateStream(n, l)
    timeStart = time.time()
    for x, s in stream:
        A += multiply_shift(x, l)
    lap = time.time() - timeStart
    print("multiply_shift sum: {}\ntime: {}\n".format(A, lap))
    
    stream = CreateStream(n, l)
    timeStart = time.time()
    for x, s in stream:
        B += multiply_mod_prime(x, q, l)
    lap = time.time() - timeStart
    print("multiply_mod_prime sum: {}\ntime: {}\n".format(B, lap))
#test(q, l)

#Opgave2
def hash_table():
    A = [ [] for _ in range(2**l) ]
    stream = CreateStream(n, l)

    for x,s in stream:
        h = multiply_shift(x, l)
        #h = multiply_mod_prime(x, q, l)
        add_h = True
        for i in range(0,len(A[h])):
            first, second = A[h][i]
            if (first == x):
                add_h = False
                A[h][i] = (x, s + second)
                exit
        if add_h:
            A[h].append((x,s))
    return A

#Opagve3
def square_sum(hash_table):
    sum = 0
    for linked_list in hash_table:
        for x, s in linked_list:
            sum += s**2
    return sum

# Del 2

In [ ]:
a0 = 0x478ad369f6852eac0de7ccf7
a1 = 0xd472f7b830cf473f771bf810
a2 = 0x695b4252f2c52ba0031649c4
a3 = 0xd4e0ab8bb4907f43b1ede881
A = [a0,a1,a2,a3]

#Opgave 4
def q_universal_hash(b, A, x):
    p = (2**b)-1 #0x01FFFFFFFFFFFFFFFFFFFFFF
    q = len(A)
    a = list(map(lambda x: p&x, A)) # this ensures each coefficient is within the range [0,p−1]

    y = a[q-1] 
    for i in range(q-2, -1, -1): # starts with the highest degree coefficient
        y = y*x + a[i] # multiplies by x and adds the next coefficient
        y = (y&p) + (y>>b)  
    if (y >= p):  
        y = y-p 
    return y

#Opgave 5
def sign_and_hash(k, b, A, x):
    #q_universal_hash(x) mod k
    g = q_universal_hash(b, A, x) 
    h = g & (k - 1)
    b = g >> (b - 1)
    s = 1 - (2*b)
    return (s,h)

#Opgave 6
class CountSketch:
    def __init__(self, t, b, A):
        self.t = t
        self.b = b
        self.A = A
        self.m = 2**t
        self.table = np.zeros(self.m, dtype=int) # Initialize a table with m zeros
    
    def update(self, x, s):
        s, h = sign_and_hash(self.m, self.b, self.A, x)
        self.table[h] += s # Update the table at position h by adding s
    
    def estimate(self):
        return np.sum(self.table**2) # Estimate by summing squares of the table entries


#Opgave 7 
def hashing_parameters(path):
    #Generate parameters for 100 random 4-universal hashfunctions:
    file = open(path,'r') ##4800 byte in hex from https://www.random.org/bytes/
    content = file.read().split()
    AA = []
    for i in range(0,len(content),12*4):
        A = []
        for j in range(0,12*4,12):
            hex_string = ""
            for k in range(12):
                hex_string += content[i+j+k]
            A.append(int(hex_string, 16))
        AA.append(A)
    return AA

def plot(n, l, b, S, path):
    AA = hashing_parameters(path)
    X = []
    for i in range(0,len(AA),1):
        stream = CreateStream(n, l)
        count_sketch = CountSketch(l, b, AA[i])
        for x, s in stream:
            count_sketch.update(x, s)
        X.append(count_sketch.estimate())
    Xx = list(range(1,101))
    Xy = sorted(X, key=lambda x: x, reverse=False)
    plt.plot(Xx, Xy, '.', label='X')

    M = []
    for i in range(0,99,11):
        G = []
        for j in range(0,11):
            G.append(X[i+j])
        G.sort(key=lambda x: x, reverse=False)
        M.append(G[5])
    Mx = list(range(6,100,11))
    My = sorted(M, key=lambda x: x, reverse=False)
    plt.plot(Mx, My, '.', label='M')
    
    plt.plot([1,100], [S,S], label='S')

    plt.legend(loc = 'upper left')
    plt.show()    

    # Calculate mean square error
    mse = np.mean([(xi - S) ** 2 for xi in X])
    return mse    

S = square_sum(hash_table())
print('n = {}'.format(n))
print('l = {}'.format(l)) 
print('S = {}'.format(S))       
mse = plot(n, l, b, S, 'RandomHex.txt')
print('Mean Square Error:', mse)
print('Var[X]:',2*S**2/m)